# 15 — Train DeepSeek-V2

**Before:** notebook **14**.

**This notebook:** PyTorch `Trainer` or optional C head-only demo.

**Learning objectives**

- Set up `Trainer` for DeepSeek-V2 on tiny Shakespeare.
- Choose PyTorch training (`RUN_TRAIN`) or C head-only demo.
- Read device and parameter count before training.
- Compare training options to notebook 8 (GPT).

**Online course:** run cells **top-to-bottom**. Setup cell must print `data OK`.

**Dojo (optional):** `dojo-grade --lesson C2-L15`


## Two training tracks

| Track | Where | What |
|-------|--------|------|
| **A — PyTorch (full model)** | This notebook | `Trainer` updates MLA + MoE + embeddings (recommended) |
| **B — C demo (head only)** | `c/bin/train_v2_tiny` | Forward full model; SGD on tied `wte`/`lm_head` only |

Full MoE+MLA **backward in C** is Phase 5 (like all of `train_gpt2.c`).


In [ ]:
# --- Setup: find repo root (llm-c-from-scratch or Cursor workbook) ---
import sys
from pathlib import Path


def find_llm_root() -> Path:
    for base in [Path.cwd(), *Path.cwd().parents]:
        if (base / "llmc" / "__init__.py").is_file():
            return base
        nested = base / "llm-c-from-scratch"
        if (nested / "llmc" / "__init__.py").is_file():
            return nested
    return Path.cwd()


ROOT = find_llm_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from llmc.notebook_utils import c_dir, checkpoint_path, data_path, v2_checkpoint_path

DATA = data_path(ROOT)
CHECKPOINT = checkpoint_path(ROOT)
C_DIR = c_dir(ROOT)
V2_CKPT = v2_checkpoint_path(ROOT)
print("ROOT", ROOT.resolve())
print("data", "OK" if DATA.is_file() else "missing")


In [ ]:
import torch
from llmc.data import CharTokenizer, load_text, train_val_split
from llmc.deepseek_v2 import DeepSeekV2, DeepSeekV2Config
from llmc.train import Trainer, TrainConfig

text = load_text(DATA)
train_text, val_text = train_val_split(text)
tok = CharTokenizer.from_text(text)
train_ids = torch.tensor(tok.encode(train_text), dtype=torch.long)
val_ids = torch.tensor(tok.encode(val_text), dtype=torch.long)

cfg = DeepSeekV2Config.tiny(tok.vocab_size, block_size=64)
model = DeepSeekV2(cfg)
device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)
print("device:", device, "| params:", f"{model.count_parameters():,}")

trainer = Trainer(
    model, train_ids, val_ids,
    TrainConfig(max_steps=200, batch_size=32, eval_interval=50, learning_rate=3e-3),
    device=device,
)


### A — PyTorch training (uncomment)


In [ ]:
RUN_TRAIN = False  # True → 200 steps on CPU (~minutes)

if RUN_TRAIN:
    history = trainer.train()
    print("last val loss:", history[-1]["val"])
else:
    print("PyTorch training skipped — set RUN_TRAIN=True (or use C track below)")


### B — C trainer (forward + optional head SGD)

```bash
cd c
make bin/train_v2_tiny
./bin/train_v2_tiny              # 10 steps, print cross-entropy
./bin/train_v2_tiny -train-head 40   # demo: loss on lm_head only
```

Read `deepseek_v2/train_v2_tiny.c` next to `llmc/train.py` — same loop shape as llm.c.


In [ ]:
RUN_C = False  # set True to compile/run C smokes

import shutil, subprocess
if RUN_C and shutil.which("make"):
    r = subprocess.run(["make", "-s", "bin/train_v2_tiny"], cwd=str(C_DIR), capture_output=True, text=True)
    if r.returncode == 0:
        r2 = subprocess.run(["./bin/train_v2_tiny"], cwd=str(C_DIR), capture_output=True, text=True)
        print(r2.stdout[-800:] if len(r2.stdout) > 800 else r2.stdout)


**Next:** notebook **16** — sample in PyTorch, then export weights for C.
